# L3 · 강도가 모자란다 — 무엇을 올릴 것인가

## 이 시간에 답할 질문

검토 결과 $\phi M_n < M_u$ 가 나왔다. 손에 쥔 선택지는 넷이다.

1. 콘크리트 강도 $f_{ck}$ 를 올린다
2. 철근 항복강도 $f_y$ 를 올린다
3. 철근을 더 넣는다 ($A_s$)
4. 단면을 키운다 ($d$)

**어느 것이 가장 잘 듣는가? 그리고 각각 무엇을 대가로 치르는가?**

학생들은 대개 1번을 먼저 떠올린다. 콘크리트 구조물이니 콘크리트를
키우면 될 것 같아서다. 이 시간에는 그 직관이 왜 틀리는지 숫자로 본다.

## 근거 조문

| 내용 | 조문 |
|---|---|
| 등가직사각형 응력블록 | KDS 14 20 20 4.1.1(8), 표 4.1-2 |
| 강도감소계수 | KDS 14 20 10 4.3.3(2) |
| 최소 휨철근량 $\phi M_n \ge 1.2 M_{cr}$ | KDS 14 20 20 4.2.2 |
| 파괴계수 $f_r = 0.63\lambda\sqrt{f_{ck}}$ | KDS 14 20 30 4.2.1 |
| 처짐을 계산하지 않아도 되는 최소 두께 | KDS 14 20 30 표 4.2-1 |

:::{tip}
같은 내용을 슬라이더로 움직여 보려면
[대화형 탐색기](../_static/explorer.html)를 연다. 값을 바꾸면 그래프가 바로
따라 바뀐다.
:::

## 0. 준비

**아래 코드가 하는 일** — 한글 글꼴을 등록하고 단면 분류 색을 정한다.

In [ ]:
%matplotlib inline

import glob
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager


def use_korean_font():
    """설치된 한글 글꼴을 찾아 matplotlib 에 등록한다."""
    site = Path(sys.prefix, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
    for pattern in (
        str(site / "koreanize_matplotlib/fonts/*.ttf"),
        "/usr/share/fonts/**/*Nanum*.ttf",
        "/usr/share/fonts/**/*NotoSansCJK*.ot[fc]",
        "/usr/share/fonts/**/*NotoSansKR*.otf",
    ):
        for path in glob.glob(pattern, recursive=True):
            font_manager.fontManager.addfont(path)

    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("NanumGothic", "Malgun Gothic", "AppleGothic",
                 "Noto Sans CJK KR", "Noto Sans KR", "WenQuanYi Zen Hei"):
        if name in installed:
            plt.rcParams["font.family"] = name
            return name

    warnings.warn(
        "한글 글꼴을 찾지 못했다. 그림의 한글이 깨진다면 "
        "`pip install koreanize-matplotlib` 로 글꼴만 내려받거나, "
        "나눔고딕·Noto Sans KR 을 시스템에 설치한다.",
        stacklevel=2,
    )
    return None


print("사용 글꼴:", use_korean_font())

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 단면 분류에 쓰는 색 (압축지배 · 변화구간 · 인장지배)
C_COMP, C_TRAN, C_TENS = "#ad3327", "#b5811f", "#2a7355"
BAND = {"압축지배단면": C_COMP, "변화구간단면": C_TRAN, "인장지배단면": C_TENS}

**아래 코드가 하는 일** — 보를 만드는 함수를 정의한다. 이번 편에서는
$f_{ck}$, $f_y$, 철근 개수, 단면 깊이를 모두 인자로 바꿀 수 있어야
하므로 함수 하나로 묶는다.

In [ ]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam(fck=27, fy=400, n_bar=4, d=600, b=400, cover=50, profile="block"):
    """단철근 직사각형 보. 압축철근이 없어 손계산과 조건이 정확히 같다."""
    kds = KDS()
    conc = kds.create_concrete_material(
        compressive_strength=fck, ultimate_profile=profile
    )
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=d, b=b,
        dia_top=22, area_top=387.1, n_top=0, c_top=cover,
        dia_bot=22, area_bot=387.1, n_bot=n_bar, c_bot=cover,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    kds.assign_concrete_section(ConcreteSection(geom))
    return kds


def column(fck=27, fy=400, column_type="tie", profile="block"):
    """500 x 500 기둥 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(
        compressive_strength=fck, ultimate_profile=profile
    )
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    kds.assign_concrete_section(ConcreteSection(geom))
    return kds


def depth(kds):
    """단면 상단에서 최하단 철근 도심까지의 유효깊이 d."""
    sec = kds.concrete_section
    top = sec.compound_geometry.geom.bounds[3]
    return top - min(g.calculate_centroid()[1] for g in sec.reinf_geometries_lumped)


print("보와 기둥을 만드는 함수를 정의했다.")

## 1. 기준 단면

**아래 코드가 하는 일** — 기준이 될 보를 만들고 주요 값을 출력한다.
400 × 600, 하부 4-D22, $f_{ck}$ 27 MPa, SD400 이다. 앞으로 이 값을
하나씩 바꿔 가며 비교한다.

In [ ]:
BASE = dict(fck=27, fy=400, n_bar=4, d=600)

def evaluate(**kwargs):
    """보 하나를 풀어 설계에 필요한 값을 한 묶음으로 돌려준다."""
    opts = {**BASE, **kwargs}
    kds = beam(**opts)
    f_res, u_res, phi = kds.ultimate_bending_capacity()
    eps_t = kds.net_tensile_strain(theta=0, d_n=u_res.d_n)
    cracked = kds.calculate_cracked_properties()
    gross = kds.get_gross_properties()
    return {
        "kds": kds,
        "d_eff": depth(kds),
        "As": opts["n_bar"] * 387.1,
        "eps_t": eps_t,
        "cls": kds.section_classification(eps_t),
        "phi": phi,
        "Mn": u_res.m_x / 1e6,
        "phiMn": f_res.m_x / 1e6,
        "Mcr": cracked.m_cr / 1e6,
        "Icr_Ig": cracked.e_ixx_c_cr / gross.e_ixx_c,
    }

base = evaluate()
print(f"단면            400 × {BASE['d']} mm, 하부 {BASE['n_bar']}-D22")
print(f"유효깊이 d      {base['d_eff']:.0f} mm")
print(f"철근량 As       {base['As']:.0f} mm²   (ρ = {base['As'] / (400 * base['d_eff']) * 100:.2f} %)")
print(f"순인장변형률 εt  {base['eps_t']:.5f}  →  {base['cls']}")
print(f"강도감소계수 φ   {base['phi']:.3f}")
print(f"공칭휨강도 Mn    {base['Mn']:.1f} kN·m")
print(f"설계휨강도 φMn   {base['phiMn']:.1f} kN·m")
print(f"균열모멘트 Mcr   {base['Mcr']:.1f} kN·m")

## 2. 하나씩 20 % 씩 올려 보기

가장 공정한 비교는 **같은 비율로 올렸을 때 강도가 몇 % 오르는가**를
보는 것이다. 공학에서는 이를 민감도 또는 탄성도라 부른다.

**아래 코드가 하는 일** — 네 변수를 각각 20 % 올려 설계휨강도의 변화를
계산한다. 철근 개수는 정수라 4 → 5 (25 %) 로 올리고, 비율을 맞춰
환산해 함께 표시한다.

In [ ]:
cases = [
    ("fck  27 → 32.4 MPa", dict(fck=32.4), 0.20),
    ("fy   400 → 480 MPa", dict(fy=480), 0.20),
    ("As   4 → 5-D22", dict(n_bar=5), 0.25),
    ("d    600 → 720 mm", dict(d=720), 0.20),
]

print(f"{'바꾼 것':22} {'φMn(kNm)':>10} {'증가':>8} {'투입 대비':>10} {'단면 분류':>12}")
print("-" * 68)
results = []
for label, kw, ratio in cases:
    r = evaluate(**kw)
    gain = r["phiMn"] / base["phiMn"] - 1
    results.append((label, r, gain, ratio))
    print(f"{label:22} {r['phiMn']:10.1f} {gain * 100:7.1f} % "
          f"{gain / ratio:9.2f} {r['cls']:>12}")

**읽는 법.** "투입 대비" 열이 탄성도다. 1.0 이면 20 % 올렸을 때 강도도
20 % 올랐다는 뜻이다.

결과의 순서가 직관과 다르다. **$f_{ck}$ 가 압도적으로 꼴찌**다.

## 3. 왜 콘크리트 강도는 휨강도에 거의 안 듣는가

손계산 식을 다시 보면 이유가 바로 보인다.

$$
M_n = A_s f_y \left(d - \frac{a}{2}\right),
\qquad a = \frac{A_s f_y}{\eta (0.85 f_{ck})\, b}
$$

$f_{ck}$ 는 **$a$ 의 분모에만** 들어 있다. 인장력 $T = A_s f_y$ 는
$f_{ck}$ 와 무관하다. 콘크리트를 강하게 하면 압축블록이 얇아져 지렛대 팔
$d - a/2$ 가 조금 길어질 뿐이다. 그런데 $a$ 는 애초에 $d$ 에 비해 작다 —
이 보에서는 68 mm 대 539 mm 다. 그 절반이 몇 mm 줄어 봐야 지렛대 팔은
1 % 도 안 늘어난다.

**휨강도는 철근이 지배한다.** 콘크리트는 압축력을 받아 주는 역할이고,
그 역할에는 이미 충분한 강도를 갖고 있다.

**아래 코드가 하는 일** — 그 사실을 그림으로 확인한다. $f_{ck}$ 를
21 부터 60 MPa 까지 올리며 $\phi M_n$, 압축블록 깊이 $a$, 지렛대 팔을
함께 그린다.

In [ ]:
fck_list = [21, 24, 27, 30, 35, 40, 50, 60]
rows = [(f, evaluate(fck=f)) for f in fck_list]

from concreteproperties_kds import stress_block_parameters
a_list = []
for f, r in rows:
    _, eta, _ = stress_block_parameters(f)
    a_list.append(r["As"] * BASE["fy"] / (eta * 0.85 * f * 400))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))

axes[0].plot(fck_list, [r["phiMn"] for _, r in rows], "o-", color="#1b4f7f", lw=2)
axes[0].set_xlabel("콘크리트 강도 fck (MPa)")
axes[0].set_ylabel("설계휨강도 φMn (kN·m)")
axes[0].set_title("fck 를 3배 가까이 올려도")
axes[0].set_ylim(0, max(r["phiMn"] for _, r in rows) * 1.15)

axes[1].plot(fck_list, a_list, "o-", color=C_COMP, lw=2, label="압축블록 깊이 a")
axes[1].plot(fck_list, [rows[0][1]["d_eff"] - a / 2 for a in a_list],
             "s-", color=C_TENS, lw=2, label="지렛대 팔 d - a/2")
axes[1].set_xlabel("콘크리트 강도 fck (MPa)")
axes[1].set_ylabel("길이 (mm)")
axes[1].set_title("정작 바뀌는 것은 이것뿐이다")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, rows[0][1]["d_eff"] * 1.1)

fig.tight_layout()
plt.show()

## 4. 그런데 $f_{ck}$ 는 다른 데 잘 듣는다

휨강도에 안 듣는다고 $f_{ck}$ 를 올리는 것이 헛일은 아니다. **강성과
균열**에는 매우 잘 듣는다.

- 탄성계수 $E_c = 8500\sqrt[3]{f_{cm}}$ — 처짐이 준다
- 파괴계수 $f_r = 0.63\lambda\sqrt{f_{ck}}$ — 균열모멘트 $M_{cr}$ 이 커져
  균열이 늦게 생긴다

설계에서 지배하는 것이 강도가 아니라 **처짐이나 균열**일 때가 많다.
특히 장경간 보나 슬래브가 그렇다. 그럴 때는 $f_{ck}$ 가 정답이 된다.

**아래 코드가 하는 일** — 세 가지를 각자의 기준값(21 MPa)으로 나눠
정규화해 한 그림에 겹친다. 배율이 1 에서 얼마나 벌어지는지 비교한다.

In [ ]:
base21 = rows[0][1]
fig, ax = plt.subplots(figsize=(7.4, 4.2))
ax.axhline(1.0, color="grey", lw=0.8, ls=":")

for key, colour, label in [
    ("phiMn", "#1b4f7f", "설계휨강도 φMn"),
    ("Mcr", C_TRAN, "균열모멘트 Mcr"),
    ("Icr_Ig", C_TENS, "균열단면 강성비 Icr/Ig"),
]:
    ax.plot(fck_list, [r[key] / base21[key] for _, r in rows],
            "o-", color=colour, lw=2, label=label)

ax.set_xlabel("콘크리트 강도 fck (MPa)")
ax.set_ylabel("fck 21 MPa 대비 배율")
ax.set_title("fck 는 강도가 아니라 강성·균열에 듣는다")
ax.legend(fontsize=9)
plt.show()

## 5. 철근을 계속 넣으면 어떻게 되는가

철근이 휨강도를 지배한다면, 그냥 계속 넣으면 되지 않을까.

여기서 [L2](L2_강도감소계수.ipynb) 의 $\phi$ 가 되돌아온다. 철근을 넣을수록
중립축이 깊어져 $\varepsilon_t$ 가 줄고, 어느 지점에서 **인장지배를
벗어나 $\phi$ 가 떨어지기 시작한다.** 그때부터는 넣은 만큼 강도가
따라오지 않는다.

**아래 코드가 하는 일** — 철근 개수를 2 부터 10 까지 늘리며 $M_n$,
$\phi M_n$, $\varepsilon_t$, $\phi$ 를 계산해 두 장으로 그린다.

In [ ]:
n_list = list(range(2, 11))
srows = [(n, evaluate(n_bar=n)) for n in n_list]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

axes[0].plot(n_list, [r["Mn"] for _, r in srows], "s--", color="grey",
             lw=1.6, label="공칭 Mn")
axes[0].plot(n_list, [r["phiMn"] for _, r in srows], "o-", color="#1b4f7f",
             lw=2.2, label="설계 φMn")
for n, r in srows:
    axes[0].plot([n], [r["phiMn"]], "o", ms=7, color=BAND[r["cls"]], zorder=3)
axes[0].set_xlabel("인장철근 개수 (D22)")
axes[0].set_ylabel("휨강도 (kN·m)")
axes[0].set_title("점 색은 단면 분류")
axes[0].legend(fontsize=9)

kds_ref = srows[0][1]["kds"]
axes[1].plot(n_list, [r["eps_t"] for _, r in srows], "o-", color="k", lw=2)
axes[1].axhline(kds_ref.eps_tl, color=C_TENS, ls="--", lw=1.2)
axes[1].axhline(kds_ref.eps_y, color=C_COMP, ls="--", lw=1.2)
axes[1].text(n_list[-1], kds_ref.eps_tl, " 인장지배 경계", va="bottom",
             ha="right", color=C_TENS, fontsize=9)
axes[1].text(n_list[-1], kds_ref.eps_y, " 압축지배 경계", va="bottom",
             ha="right", color=C_COMP, fontsize=9)
for n, r in srows:
    axes[1].plot([n], [r["eps_t"]], "o", ms=7, color=BAND[r["cls"]], zorder=3)
axes[1].set_xlabel("인장철근 개수 (D22)")
axes[1].set_ylabel("순인장변형률 εt")
axes[1].set_title("철근을 넣을수록 εt 가 준다")

fig.tight_layout()
plt.show()

**아래 코드가 하는 일** — 철근 하나를 더 넣을 때마다 강도가 얼마나
늘어나는지, 그 증분을 표로 본다. 수익 체감이 어디서 시작되는지 확인한다.

In [ ]:
print(f"{'철근':>8} {'εt':>9} {'φ':>7} {'φMn':>9} {'직전 대비 증가':>14} {'단면 분류':>12}")
print("-" * 66)
prev = None
for n, r in srows:
    inc = "—" if prev is None else f"{r['phiMn'] - prev:+.1f} kN·m"
    print(f"{n:6d}-D22 {r['eps_t']:9.5f} {r['phi']:7.3f} {r['phiMn']:9.1f} "
          f"{inc:>14} {r['cls']:>12}")
    prev = r["phiMn"]

**여기서 알 것.** 인장지배를 유지하는 동안은 철근 하나당 증가폭이 거의
일정하다. 변화구간에 들어서면 증가폭이 눈에 띄게 줄어든다.

**설계자가 읽어야 할 신호.** 증가폭이 꺾이기 시작했다면 그것은
"철근을 더 넣지 말고 **단면을 키우라**"는 뜻이다. 기준이 직접 그렇게
말하지는 않지만, $\phi$ 를 통해 경제적 유인을 만들어 그렇게 유도한다.
이것이 강도감소계수의 숨은 역할이다 — 안전만이 아니라 **바람직한 설계
형태를 유도**한다.

## 6. 단면 깊이가 가장 잘 듣는다

2절에서 $d$ 의 탄성도가 1 을 넘었다. 이유는 $M_n = A_s f_y (d - a/2)$
에서 $d$ 가 **직접** 곱해지기 때문이다. 게다가 $d$ 를 키우면
$\varepsilon_t$ 도 커져서 $\phi$ 가 유지되거나 오른다. 두 효과가 같은
방향으로 작용한다.

**아래 코드가 하는 일** — 단면 깊이를 500 부터 800 mm 까지 바꾸며
$\phi M_n$ 과 $\varepsilon_t$ 를 함께 본다.

In [ ]:
d_list = [500, 550, 600, 650, 700, 750, 800]
drows = [(d, evaluate(d=d)) for d in d_list]

fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.plot(d_list, [r["phiMn"] for _, r in drows], "o-", color="#1b4f7f", lw=2.2)
ax.set_xlabel("단면 깊이 h (mm)")
ax.set_ylabel("설계휨강도 φMn (kN·m)")
ax.set_title("깊이를 키우면 강도는 거의 비례해 오른다")

ax2 = ax.twinx()
ax2.plot(d_list, [r["eps_t"] for _, r in drows], "s--", color=C_TENS, lw=1.6)
ax2.set_ylabel("순인장변형률 εt", color=C_TENS)
ax2.tick_params(axis="y", labelcolor=C_TENS)
ax2.grid(False)

ax.text(d_list[1], max(r["phiMn"] for _, r in drows) * 0.35,
        "파랑: φMn (왼쪽 축)\n초록 점선: εt (오른쪽 축)", fontsize=9)
plt.show()

**대가는 무엇인가.** 자중이 늘고, 층고가 커지고, 공사비가 오른다.
보 깊이는 대개 건축 계획이 먼저 정해 놓는 값이라, 구조기술자가 마음대로
바꿀 수 없는 경우가 많다. **가장 잘 듣는 변수가 가장 못 바꾸는 변수**인
것이 실무의 현실이다.

## 7. 설계의 의도 — 왜 최소 두께 표가 따로 있는가

KDS 14 20 30 표 4.2-1 은 **처짐을 계산하지 않아도 되는 최소 두께**를
규정한다. 단순지지 보는 $\ell/16$, 1단 연속은 $\ell/18.5$ 같은 식이다.

왜 강도와 별개로 이런 규정을 둘까. 4절에서 본 것처럼 **처짐을 지배하는
것과 강도를 지배하는 것이 다르기** 때문이다. 강도는 철근이, 처짐은 강성
$E_c I$ 가, 즉 사실상 **단면 깊이의 세제곱**이 지배한다. 철근을 아무리
넣어도 처짐은 별로 안 줄어든다.

그래서 기준은 순서를 정해 준다.

1. 먼저 최소 두께로 **깊이를 정한다** (사용성)
2. 그 깊이에서 필요한 **철근을 계산한다** (강도)

이 순서를 뒤집어 강도만 보고 얇은 보를 설계하면, 강도 검토는 통과하고
처짐 검토에서 걸린다. 그때는 이미 다른 것이 다 정해진 뒤라 되돌리기가
어렵다.

**아래 코드가 하는 일** — 최소 두께 규정이 실제로 어떤 깊이를 요구하는지
경간별로 확인한다.

In [ ]:
from concreteproperties_kds import minimum_thickness

supports = ("단순지지", "1단연속", "양단연속", "캔틸레버")

print(f"{'경간(m)':>8}" + "".join(f"{s:>11}" for s in supports))
print("-" * 52)
for span in (4, 6, 8, 10, 12):
    vals = [minimum_thickness(span=span * 1000, member="보", support=s, fy=400)
            for s in supports]
    print(f"{span:8.0f} " + " ".join(f"{v:9.0f}mm" for v in vals))

## 8. 직접 바꿔 보기

**아래 코드가 하는 일** — 2절의 민감도 비교를 원하는 조건으로 다시
돌린다. 기준 단면과 증가율을 바꿔 가며, 순위가 뒤집히는 경우가 있는지
찾아보라.

In [ ]:
BASE = dict(fck=27, fy=400, n_bar=8, d=600)   # ← n_bar 를 4 에서 8 로 바꿨다
base = evaluate()

print(f"기준: fck {BASE['fck']}, SD{BASE['fy']}, {BASE['n_bar']}-D22, h {BASE['d']} mm")
print(f"      φMn = {base['phiMn']:.1f} kN·m,  εt = {base['eps_t']:.5f}"
      f"  →  {base['cls']}\n")

print(f"{'바꾼 것':22} {'φMn(kNm)':>10} {'증가':>8} {'투입 대비':>10} {'단면 분류':>12}")
print("-" * 68)
for label, kw, ratio in [
    ("fck  +20 %", dict(fck=BASE["fck"] * 1.2), 0.20),
    ("fy   +20 %", dict(fy=min(600, BASE["fy"] * 1.2)), 0.20),
    ("As   +25 %", dict(n_bar=BASE["n_bar"] + 2), 0.25),
    ("d    +20 %", dict(d=int(BASE["d"] * 1.2)), 0.20),
]:
    r = evaluate(**kw)
    gain = r["phiMn"] / base["phiMn"] - 1
    print(f"{label:22} {r['phiMn']:10.1f} {gain * 100:7.1f} % "
          f"{gain / ratio:9.2f} {r['cls']:>12}")

BASE = dict(fck=27, fy=400, n_bar=4, d=600)   # 기준을 되돌린다

## 9. 생각해 볼 문제

1. **8절에서 철근을 8-D22 로 늘린 뒤 민감도 순위가 어떻게 달라졌는가?**
   특히 $A_s$ 의 탄성도가 왜 떨어졌는지 $\phi$ 와 연결해 설명하라.

2. **$f_y$ 를 올리는 것은 $A_s$ 를 늘리는 것과 수식상 같다**
   ($T = A_s f_y$). 그런데 설계에서 두 선택은 같지 않다. 무엇이
   다른가? ([L2](L2_강도감소계수.ipynb) 10절 1번 문제와 함께 생각하라.)

3. **경간 10 m 단순지지 보를 설계한다.** 표에서 최소 두께가 625 mm 로
   나왔다. 그런데 건축 계획상 500 mm 밖에 쓸 수 없다면 어떻게 하겠는가?
   KDS 14 20 30 은 이 경우 무엇을 요구하는가?

4. **"콘크리트 강도를 올려도 휨강도는 거의 안 오른다"는 결론이 항상
   성립하는가?** 압축철근이 많은 복철근 보나, 축력을 함께 받는 기둥에서는
   어떨까? 이유를 들어 예상한 뒤 코드로 확인하라.

## 정리

| 올리는 것 | 휨강도 | 처짐·균열 | 대가 |
|---|---|---|---|
| $f_{ck}$ | 거의 안 듦 | **잘 듦** | 배합·품질관리 |
| $f_y$ | 잘 듦 | 안 듦 | 인장지배 유지가 어려워짐 |
| $A_s$ | 잘 듦, 그러나 **수익 체감** | 조금 듦 | 배근 간격·정착 |
| $d$ | **가장 잘 듦** | **가장 잘 듦** | 자중·층고·공사비 |

- 휨강도는 철근이 지배한다. 콘크리트 강도는 지렛대 팔에만 관여한다.
- $f_{ck}$ 는 강성과 균열에 듣는다. 처짐이 지배하는 부재에서는 이쪽이
  정답이다.
- 철근의 수익 체감은 $\phi$ 가 만든다. 증가폭이 꺾이면 단면을 키우라는
  신호다.
- 기준이 최소 두께를 먼저 정하게 한 것은, 처짐과 강도의 지배 인자가
  다르기 때문이다.

조문과 구현 함수의 대응은
[설계식 목록](../user_guide/design_codes/equations.md) 에 정리되어 있다.